In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [17]:
# Reading in data here

US_ng_data = pd.read_csv("data/us_monthly_NG_prices.csv", parse_dates=['Date'])

SG_ng_data = pd.read_csv("data/asia_monthly_NG_prices.csv", parse_dates=['observation_date'])
SG_ng_data.rename(columns={'PNGASJPUSDM': 'Price'}, inplace=True)

In [18]:
US_ng_data.head()
SG_ng_data.head()

,observation_date,Price
0,2016-02-01,7.99
1,2016-03-01,8.01
2,2016-04-01,6.68
3,2016-05-01,6.81
4,2016-06-01,7.08


In [22]:
# Ensure 'date' is in datetime format (if not already)
US_ng_data['Date'] = pd.to_datetime(US_ng_data['Date'])

# Extract the year from the date
US_ng_data['year'] = US_ng_data['Date'].dt.year

# Group by year and calculate the average price
avg_price_per_year_us = US_ng_data.groupby('year')['Price'].mean().reset_index()

# Rename columns for clarity (optional)
avg_price_per_year_us.columns = ['Year', 'Price_US']

# Display the new DataFrame
print(avg_price_per_year_us)

    Year  Price_US
0   1997  2.496667
1   1998  2.090833
2   1999  2.270000
3   2000  4.309167
4   2001  3.956667
5   2002  3.366667
6   2003  5.485833
7   2004  5.900000
8   2005  8.811667
9   2006  6.745000
10  2007  6.976667
11  2008  8.861667
12  2009  3.948333
13  2010  4.386667
14  2011  4.000000
15  2012  2.752500
16  2013  3.728333
17  2014  4.391667
18  2015  2.630000
19  2016  2.515000
20  2017  2.985833
21  2018  3.166667
22  2019  2.565833
23  2020  2.033333
24  2021  3.908333
25  2022  6.418333
26  2023  2.535833
27  2024  2.193333
28  2025  3.526667
29  2026  5.670000


In [23]:
# Ensure 'date' is in datetime format (if not already)
SG_ng_data['observation_date'] = pd.to_datetime(SG_ng_data['observation_date'])

# Extract the year from the date
SG_ng_data['year'] = SG_ng_data['observation_date'].dt.year

# Group by year and calculate the average price
avg_price_per_year_sg = SG_ng_data.groupby('year')['Price'].mean().reset_index()

# Rename columns for clarity (optional)
avg_price_per_year_sg.columns = ['Year', 'Price_SG']

# Display the new DataFrame
print(avg_price_per_year_sg)

    Year   Price_SG
0   2016   7.346364
1   2017   7.247445
2   2018   9.795415
3   2019   5.444624
4   2020   4.366424
5   2021  18.600467
6   2022  33.297022
7   2023  13.463505
8   2024  11.708527
9   2025  12.087750
10  2026  10.593500


In [24]:
combined_df = pd.merge(avg_price_per_year_us, avg_price_per_year_sg, on='Year', how='outer')
print(combined_df)

    Year  Price_US   Price_SG
0   1997  2.496667        NaN
1   1998  2.090833        NaN
2   1999  2.270000        NaN
3   2000  4.309167        NaN
4   2001  3.956667        NaN
5   2002  3.366667        NaN
6   2003  5.485833        NaN
7   2004  5.900000        NaN
8   2005  8.811667        NaN
9   2006  6.745000        NaN
10  2007  6.976667        NaN
11  2008  8.861667        NaN
12  2009  3.948333        NaN
13  2010  4.386667        NaN
14  2011  4.000000        NaN
15  2012  2.752500        NaN
16  2013  3.728333        NaN
17  2014  4.391667        NaN
18  2015  2.630000        NaN
19  2016  2.515000   7.346364
20  2017  2.985833   7.247445
21  2018  3.166667   9.795415
22  2019  2.565833   5.444624
23  2020  2.033333   4.366424
24  2021  3.908333  18.600467
25  2022  6.418333  33.297022
26  2023  2.535833  13.463505
27  2024  2.193333  11.708527
28  2025  3.526667  12.087750
29  2026  5.670000  10.593500


In [27]:
# Rename columns for clarity (optional, based on your description)
combined_df.rename(columns={'PriceA': 'Price_US', 'PriceB': 'Price_SG'}, inplace=True)

# Remove all rows with NAs
combined_df.dropna(inplace=True)

# Regression with intercept (includes a constant term)
X_with_intercept = sm.add_constant(combined_df['Price_SG'])  # Predictor with intercept
model_with_intercept = sm.OLS(combined_df['Price_US'], X_with_intercept).fit()  # Response: Price_US
print("Regression Model WITH Intercept:")
print(model_with_intercept.summary())

# Regression without intercept (no constant term)
model_without_intercept = sm.OLS(combined_df['Price_US'], combined_df['Price_SG']).fit()  # Response: Price_US, Predictor: Price_SG
print("\nRegression Model WITHOUT Intercept:")
print(model_without_intercept.summary())

Regression Model WITH Intercept:
                            OLS Regression Results                            
Dep. Variable:               Price_US   R-squared:                       0.559
Model:                            OLS   Adj. R-squared:                  0.510
Method:                 Least Squares   F-statistic:                     11.42
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00813
Time:                        20:02:04   Log-Likelihood:                -14.475
No. Observations:                  11   AIC:                             32.95
Df Residuals:                       9   BIC:                             33.74
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.80